# Exercises

This is the notebook containing the exercises for Feature Store, Model Monitor, and Clarify. Tested for these exercises was performed using __2 vCPU + 4 GiB notebook instance with Python 3 (TensorFlow 2.1 Python 3.6 CPU Optimized) kernel__.

## Staging

We'll begin by initializing some variables. These are often assumed to be present in code samples you'll find in the AWS documenation.

In [1]:
# import sagemaker
# from sagemaker.session import Session
# from sagemaker import get_execution_role

# role = get_execution_role()
# session = sagemaker.Session()
# region = session.boto_region_name
# bucket = session.default_bucket()

In [2]:
from sagemaker.core.helper.session_helper import Session, get_execution_role

role = get_execution_role()
session = Session()
region = session.boto_region_name
bucket = session.default_bucket()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


## Feature Store
---

Feature store is a special database to give ML systems a consistent data flow across training and inference workloads. It can ingest data in batches (for training) as well as serve input features to models with very low latency for real-time prediction.

For this exercise we'll work with a wine quality dataset: https://archive.ics.uci.edu/ml/datasets/wine+quality/

```P. Cortez, A. Cerdeira, F. Almeida, T. Matos and J. Reis.
Modeling wine preferences by data mining from physicochemical properties. In Decision Support Systems, Elsevier, 47(4):547-553, 2009.```

In [5]:
import pandas as pd
from sklearn import datasets
import time
import uuid

In [6]:
data = datasets.load_wine()
df = pd.DataFrame(data['data'])
df.columns = data['feature_names']

In [5]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


If we leave the column names as-is, Feature Store won't be able to handle the `/` in `od280/od315_of_diluted_wines` (`/` is a delimiter Feature Store uses to manage how features are organized.)

In [6]:
df.rename(columns = {'od280/od315_of_diluted_wines':'od280_od315_of_diluted_wines'}, inplace=True)

In [7]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


Once we have our data, we can create a feature group. Remember to attach event time and ID columns - Feature Store needs them.

In [8]:
# Build the definitions from the DataFrame yourself

def to_feature_type(dtype):
    name = str(dtype)
    if name.startswith("int"):
        return "Integral"
    if name.startswith("float"):
        return "Fractional"
    if name == "object":
        return "String"
    raise ValueError(f"Unsupported dtype: {name}")

In [22]:
# Add a feature group
df["EventTime"] = time.time()
df["id"] = range(len(df))

# Previous existing code is deprecated and was deleted

# Create FeatureDefinition object after adding the EventTime and id columns
from sagemaker.core.shapes import FeatureDefinition

feature_definitions = [
    FeatureDefinition(
        feature_name=col,
        feature_type=to_feature_type(df[col].dtype),
    )
    for col in df.columns
]
feature_definitions

[FeatureDefinition(feature_name='alcohol', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='malic_acid', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='ash', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='alcalinity_of_ash', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='magnesium', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='total_phenols', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='flavanoids', feature_type='Fractional', collection_type=Unassigned(), collection_config=Unassigned()),
 FeatureDefinition(feature_name='nonflavanoid_phe

In [23]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline,EventTime,id
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,1.791103e+09,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,1.791103e+09,1
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,1.791103e+09,2
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,1.791103e+09,3
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,1.791103e+09,4


The feature group is not created until we call the `create` method, let's do that now:

Create feature group

In [24]:
from sagemaker.core.shapes import OnlineStoreConfig, OfflineStoreConfig,S3StorageConfig
from sagemaker.core.resources import FeatureGroup


# Create feature group 
feature_group = FeatureGroup.create(
    feature_group_name="wine-quality-features",
    record_identifier_feature_name="id",
    event_time_feature_name="EventTime",
    feature_definitions=feature_definitions,
    role_arn=role,
    online_store_config=OnlineStoreConfig(enable_online_store=True),  # To serve inferences in real time
    offline_store_config=OfflineStoreConfig(
        s3_storage_config=S3StorageConfig(
            s3_uri=f"s3://{bucket}/lesson4/features",
        ),
    ),
    region=region,
)

[10/04/26 08:38:07] INFO     Creating feature_group resource.                                    ]8;id=4738475;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4738476;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11769\11769]8;;\

Lastly, ingest some data into your feature group:

In [30]:
# TODO
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

runtime = session.boto_session.client(
    'sagemaker-featurestore-runtime',
    region_name=region
)

records = df.to_dict(orient="records")

def put_row(rec):
    runtime.put_record(
        FeatureGroupName="wine-quality-features",
        Record=[
            {"FeatureName": col, "ValueAsString": str(val)}
            for col, val in rec.items()
        ],
    )

with ThreadPoolExecutor(max_workers=3) as ex:
    list(tqdm(ex.map(put_row, records), total=len(records)))

100%|██████████| 178/178 [00:00<00:00, 202.03it/s]


In [36]:
# testing the fetching of data from the FeatureGroup Database

print(df.iloc[1])
print("-----------------------")
data = runtime.get_record(
    FeatureGroupName="wine-quality-features",
    RecordIdentifierValueAsString="1"
)
print(data["Record"])

alcohol                         1.320000e+01
malic_acid                      1.780000e+00
ash                             2.140000e+00
alcalinity_of_ash               1.120000e+01
magnesium                       1.000000e+02
total_phenols                   2.650000e+00
flavanoids                      2.760000e+00
nonflavanoid_phenols            2.600000e-01
proanthocyanins                 1.280000e+00
color_intensity                 4.380000e+00
hue                             1.050000e+00
od280_od315_of_diluted_wines    3.400000e+00
proline                         1.050000e+03
EventTime                       1.791103e+09
id                              1.000000e+00
Name: 1, dtype: float64
-----------------------
[{'FeatureName': 'alcohol', 'ValueAsString': '13.2'}, {'FeatureName': 'malic_acid', 'ValueAsString': '1.78'}, {'FeatureName': 'ash', 'ValueAsString': '2.14'}, {'FeatureName': 'alcalinity_of_ash', 'ValueAsString': '11.2'}, {'FeatureName': 'magnesium', 'ValueAsString': '100.0'},

Great job! You've demonstrated your understanding of creating feature groups and ingesting data into them using Feature Store. Next up we'll cover Model Monitor!

## Model Monitor

In this exercise we'll create a monitoring schedule for a deployed model. We're going to provide code to help you deploy a model and get started, so that you can focus on Model Monitor for this exercise. __Remember to clean up your model before you end a work session__. We'll provide some code at the end to help you clean up your model. We'll begin by reloading our data from the previous exercise.



In [11]:
data = datasets.load_wine()
df = pd.DataFrame(data['data'])
df.columns = data['feature_names']
df.rename(columns = {'od280/od315_of_diluted_wines':'od280_od315_of_diluted_wines'}, inplace=True)

We also need to put the target variable in the first column per the docs for our chosen algorithm: https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html

In [12]:
df["TARGET"] = data['target']
df.set_index(df.pop('TARGET'), inplace=True)
df.reset_index(inplace=True)

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   TARGET                        178 non-null    int64  
 1   alcohol                       178 non-null    float64
 2   malic_acid                    178 non-null    float64
 3   ash                           178 non-null    float64
 4   alcalinity_of_ash             178 non-null    float64
 5   magnesium                     178 non-null    float64
 6   total_phenols                 178 non-null    float64
 7   flavanoids                    178 non-null    float64
 8   nonflavanoid_phenols          178 non-null    float64
 9   proanthocyanins               178 non-null    float64
 10  color_intensity               178 non-null    float64
 11  hue                           178 non-null    float64
 12  od280_od315_of_diluted_wines  178 non-null    float64
 13  proli

Now we'll upload the data to S3 as train and validation data:

In [14]:
from sagemaker.core.training.configs import InputData  # Added since sagemaker.inputs.TrainingInput() is deprecated

delimiter = int(len(df)/2)
train, test = df.iloc[delimiter:], df.iloc[:delimiter]

train.to_csv("train.csv", header=False, index=False)
test.to_csv("validation.csv", header=False, index=False)

val_location = session.upload_data('./validation.csv', key_prefix="data")
train_location = session.upload_data('./train.csv', key_prefix="data")

# s3_input_train = sagemaker.inputs.TrainingInput(s3_data=train_location, content_type='csv')
# s3_input_validation = sagemaker.inputs.TrainingInput(s3_data=val_location, content_type='csv')

s3_input_train = InputData(channel_name="train", data_source=train_location, content_type="text/csv")
s3_input_validation = InputData(channel_name="validation", data_source=val_location, content_type="text/csv")

In [15]:
# algo_image = sagemaker.image_uris.retrieve("xgboost", region, version='latest')
# s3_output_location = f"s3://{bucket}/models/wine_model"

# model=sagemaker.estimator.Estimator(
#     image_uri=algo_image,
#     role=role,
#     instance_count=1,
#     instance_type='ml.m4.xlarge',
#     volume_size=5,
#     output_path=s3_output_location,
#     sagemaker_session=sagemaker.Session()
# )

# model.set_hyperparameters(max_depth=5,
#                         eta=0.2,
#                         gamma=4,
#                         min_child_weight=6,
#                         subsample=0.8,
#                         objective='reg:linear',
#                         early_stopping_rounds=10,
#                         num_round=200)


# model.fit({'train': s3_input_train, 'validation': s3_input_validation})

from sagemaker.core import image_uris
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import Compute, OutputDataConfig

algo_image = image_uris.retrieve("xgboost", region, version='latest')
s3_output_location = f"s3://{bucket}/models/wine_model"

model = ModelTrainer(
    training_image=algo_image,
    role=role,
    compute=Compute(
        instance_type="ml.m5.xlarge",
        instance_count=1,
        volume_size_in_gb=5,
    ),
    output_data_config=OutputDataConfig(s3_output_path=s3_output_location),
    hyperparameters={
        "max_depth": "5",
        "eta": "0.2",
        "gamma": "4",
        "min_child_weight": "6",
        "subsample": "0.8",
        "objective": "reg:linear",
        "early_stopping_rounds": "10",
        "num_round": "200",
    },
)

model.train(
    input_data_config=[s3_input_train, s3_input_validation],
    wait=True,
)

[10/06/26 09:22:52] INFO     Ignoring unnecessary instance type: None.                            ]8;id=14743973;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=14743974;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#535\535]8;;\

[10/06/26 09:22:53] INFO     SageMaker session not provided. Using default Session.                  ]8;id=14743981;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=14743982;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     Base name not provided. Using default name:                             ]8;id=14743988;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=14743989;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#90\90]8;;\
                             xgboost-job                                                                           

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=14743995;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=14743996;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=14744002;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=14744003;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

...

                    INFO     xgboost-job-20261006092253/algo-1-1791278617:                       ]8;id=14744227;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744228;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [09:24:42] src/tree/updater_prune.cc:74: tree pruning end, 1 roots,                   
                             0 extra nodes, 6 pruned nodes, max_depth=0                                            

                    INFO     xgboost-job-20261006092253/algo-1-1791278617:                       ]8;id=14744233;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744234;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [10]#011train-rmse:0.264732#011validation-rmse:1.1432                                 

                    INFO     xgboost-job-20261006092253/algo-1-1791278617:                       ]8;id=14744239;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744240;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Stopping. Best iteration:                                                             

                    INFO     xgboost-job-20261006092253/algo-1-1791278617:                       ]8;id=14744245;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744246;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             [0]#011train-rmse:0.9357#011validation-rmse:0.53422                                   

[10/06/26 09:25:05] INFO     Final Resource Status: Completed                                    ]8;id=14744252;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744253;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31590\31590]8;;\

Now that your training job has finished, you can perform the first task in this exercise: creating a data capture config. Configure your model to sample `34%` of inferences:

In [16]:
# TODO
# from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.core.shapes import DataCaptureConfig, CaptureOption

capture_uri = f's3://{bucket}/lesson4/exercice2/data-capture'

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    initial_sampling_percentage=34,
    destination_s3_uri=capture_uri,
    capture_options=[
        CaptureOption(capture_mode="Input"),
        CaptureOption(capture_mode="Output"),
    ],
)

Great! We'll use your config to deploy a model below:

In [17]:
# xgb_predictor = model.deploy( <- deprecated
#     initial_instance_count=1, instance_type='ml.m4.xlarge',
#     data_capture_config=data_capture_config
# )

from sagemaker.core.resources import Model, EndpointConfig, Endpoint
from sagemaker.core.shapes import ContainerDefinition, ProductionVariant

model_data = model._latest_training_job.model_artifacts.s3_model_artifacts

# Tear down in dependency order: endpoint -> config -> model
try:
    ep = Endpoint.get(endpoint_name="wine-endpoint")
    ep.delete()
    ep.wait_for_delete()
except Exception:
    pass

try:
    EndpointConfig.get(endpoint_config_name="wine-config").delete()
except Exception:
    pass

try:
    Model.get(model_name="wine-model").delete()
except Exception:
    pass

Model.create(
    model_name="wine-model",
    primary_container=ContainerDefinition(image=algo_image, model_data_url=model_data),
    execution_role_arn=role,
)

EndpointConfig.create(
    endpoint_config_name="wine-config",
    production_variants=[ProductionVariant(
        variant_name="default",
        model_name="wine-model",
        instance_type="ml.m5.xlarge",
        initial_instance_count=1,
    )],
    data_capture_config=data_capture_config,
)

# NOTE: Endpoint.create succeeds (CreateEndpoint is called), but its return
# value blows up: it ends with cls.get(), and the generated Pydantic model
# marks data_capture_config.kms_key_id as required while AWS omits KmsKeyId
# when no KMS key is set. Any resource-layer call that describes this endpoint
# (get / wait_for_status / refresh / invoke) hits the same error.
# Workaround: create + poll + invoke via boto3 for endpoints with data capture.

# xgb_predictor = Endpoint.create(
#     endpoint_name="wine-endpoint",
#     endpoint_config_name="wine-config",
# )

# xgb_predictor.wait_for_status("InService")

import boto3

sm = boto3.client("sagemaker")
sm.create_endpoint(
    EndpointName="wine-endpoint",
    EndpointConfigName="wine-config",
)
sm.get_waiter("endpoint_in_service").wait(EndpointName="wine-endpoint")

                    INFO     Creating model resource.                                            ]8;id=14744265;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744266;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#20592\20592]8;;\

[10/06/26 09:25:06] INFO     Creating endpoint_config resource.                                  ]8;id=14744272;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744273;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#11068\11068]8;;\

Great! You should see an indicator like this when the deployment finishes:

```
-----------------!
```
We can test your deployment like so:

In [16]:
# xgb_predictor.serializer = sagemaker.serializers.CSVSerializer() <- deprecated. We have to do the serialization by ourselves
# inputs = test.copy()
# # Drop the target variable
# inputs = inputs.drop(columns=inputs.columns[0])
# x_pred = xgb_predictor.predict(inputs.sample(5).values).decode('utf-8')
# x_pred

import io, csv

inputs = test.copy()
inputs = inputs.drop(columns=inputs.columns[0])

buf = io.StringIO()
csv.writer(buf).writerows(inputs.sample(5).values)

# resp = xgb_predictor.invoke(body=buf.getvalue(), content_type="text/csv")  <- In this example, we used boto3 to create the endpoint instead of xgb_predictor = Endpoint.create(). So xgb_predictor does not exist. So we will use boto3 to invoke the endpoint instead.
# x_pred = resp.body.read().decode("utf-8") <- didn't delete it since it would still be useful in case we used xgb_predictor = Endpoint.create() and used the commented out resp variable above.

# Using equivalent of above but with boto3 

rt = boto3.client("sagemaker-runtime")
resp = rt.invoke_endpoint(
    EndpointName="wine-endpoint",
    ContentType="text/csv",
    Body=buf.getvalue(),
)

x_pred = resp["Body"].read().decode("utf-8")
x_pred

'0.6030303239822388,0.6030303239822388,0.6030303239822388,0.6030303239822388,0.6030303239822388'

All systems go! To finish up the exercise, we're going to provide you with a DefaultModelMonitor and a suggested baseline. Combine the `xgb_predictor` and the provided `my_monitor` to configure the monitoring schedule for _hourly_ monitoring.

In [ ]:
# from sagemaker.model_monitor import DefaultModelMonitor
# from sagemaker.model_monitor.dataset_format import DatasetFormat

# my_monitor = DefaultModelMonitor(  <- deprecated. DefaultModelMonitor doesn't exist and there's no object to hold the config; it moves into the processing job below.
#     role=role,
#     instance_count=1,
#     instance_type='ml.m5.xlarge',
#     volume_size_in_gb=20,
#     max_runtime_in_seconds=3600,
# )

In [20]:
# my_monitor.suggest_baseline(
#     baseline_dataset=f's3://{bucket}/data/train.csv',
#     dataset_format=DatasetFormat.csv(header=False),
# )

from sagemaker.core.resources import ProcessingJob
from sagemaker.core.shapes import (
    ProcessingInput, ProcessingS3Input, ProcessingOutput, ProcessingS3Output,
    ProcessingOutputConfig, ProcessingResources, ProcessingClusterConfig,
    AppSpecification, ProcessingStoppingCondition,
)
import time
job_name = f"wine-baseline-{int(time.time())}"

monitor_image = f"156813124566.dkr.ecr.{region}.amazonaws.com/sagemaker-model-monitor-analyzer"
baseline_results_uri = f"s3://{bucket}/lesson4/exercice2/monitoring/baseline"

job = ProcessingJob.create(
    processing_job_name=job_name,
    app_specification=AppSpecification(image_uri=monitor_image),
    role_arn=role,
    environment={
        "dataset_format": '{"csv": {"header": false}}',
        "dataset_source": "/opt/ml/processing/input/baseline_dataset_input",
        "output_path": "/opt/ml/processing/output",
        "publish_cloudwatch_metrics": "Disabled",
        "analysis_type": "DATA_QUALITY",
    },
    processing_inputs=[ProcessingInput(
        input_name="baseline_dataset_input",
        s3_input=ProcessingS3Input(
            s3_uri=f"s3://{bucket}/data/train.csv",
            local_path="/opt/ml/processing/input/baseline_dataset_input",
            s3_data_type="S3Prefix",
            s3_input_mode="File",
            s3_data_distribution_type="FullyReplicated",
        ),
    )],
    processing_output_config=ProcessingOutputConfig(outputs=[ProcessingOutput(
        output_name="monitoring_output",
        s3_output=ProcessingS3Output(
            s3_uri=baseline_results_uri,
            local_path="/opt/ml/processing/output",
            s3_upload_mode="EndOfJob",
        ),
    )]),
    processing_resources=ProcessingResources(
        cluster_config=ProcessingClusterConfig(
            instance_count=1,
            instance_type="ml.m5.xlarge",
            volume_size_in_gb=20,
        )
    ),
    stopping_condition=ProcessingStoppingCondition(max_runtime_in_seconds=3600),
)

[10/05/26 09:53:03] INFO     Creating processing_job resource.                                   ]8;id=2518080;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2518081;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#28846\28846]8;;\

In [23]:
job.refresh()
print(job.processing_job_status)
print(job.failure_reason)

Completed



Below, provide the monitoring schedule:

In [25]:
# # TODO
# from sagemaker.model_monitor import CronExpressionGenerator <- deprecated

# my_monitor.create_monitoring_schedule(
# # ...
# )

# from sagemaker.model_monitor import CronExpressionGenerator

# my_monitor.create_monitoring_schedule(
#     monitor_schedule_name='my-monitoring-schedule',
#     endpoint_input=xgb_predictor.endpoint_name,
#     statistics=my_monitor.baseline_statistics(),
#     constraints=my_monitor.suggested_constraints(),
#     schedule_cron_expression=CronExpressionGenerator.hourly(),
# )

from sagemaker.core.resources import MonitoringSchedule
from sagemaker.core.shapes import (
    MonitoringScheduleConfig, ScheduleConfig, MonitoringJobDefinition,
    MonitoringInput, EndpointInput, MonitoringOutputConfig, MonitoringOutput,
    MonitoringS3Output, MonitoringResources, MonitoringClusterConfig,
    MonitoringAppSpecification, MonitoringStoppingCondition,
    MonitoringBaselineConfig, MonitoringStatisticsResource,
    MonitoringConstraintsResource,
)

MonitoringSchedule.create(
    monitoring_schedule_name="my-wine-endpoint-monitoring-schedule",
    monitoring_schedule_config=MonitoringScheduleConfig(
        schedule_config=ScheduleConfig(schedule_expression="cron(0 * ? * * *)"),
        monitoring_job_definition=MonitoringJobDefinition(
            baseline_config=MonitoringBaselineConfig(
                statistics_resource=MonitoringStatisticsResource(
                    s3_uri=f"{baseline_results_uri}/statistics.json"),
                constraints_resource=MonitoringConstraintsResource(
                    s3_uri=f"{baseline_results_uri}/constraints.json"),
            ),
            monitoring_inputs=[MonitoringInput(
                endpoint_input=EndpointInput(
                    endpoint_name="wine-endpoint",
                    local_path="/opt/ml/processing/input/endpoint",
                )
            )],
            monitoring_output_config=MonitoringOutputConfig(
                monitoring_outputs=[MonitoringOutput(
                    s3_output=MonitoringS3Output(
                        s3_uri=f"s3://{bucket}/monitoring/reports",
                        local_path="/opt/ml/processing/output",
                    )
                )]
            ),
            monitoring_resources=MonitoringResources(
                cluster_config=MonitoringClusterConfig(
                    instance_count=1,
                    instance_type="ml.m5.xlarge",
                    volume_size_in_gb=20,
                )
            ),
            monitoring_app_specification=MonitoringAppSpecification(
                image_uri=monitor_image,
            ),
            stopping_condition=MonitoringStoppingCondition(
                max_runtime_in_seconds=3600,
            ),
            role_arn=role,
        ),
    ),
)

[10/05/26 09:58:35] INFO     Creating monitoring_schedule resource.                              ]8;id=2518093;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=2518094;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#24562\24562]8;;\

MonitoringSchedule(monitoring_schedule_name='my-wine-endpoint-monitoring-schedule', monitoring_schedule_arn='arn:aws:sagemaker:us-east-1:864107284170:monitoring-schedule/my-wine-endpoint-monitoring-schedule', monitoring_schedule_status='Pending', monitoring_type='DataQuality', failure_reason=Unassigned(), creation_time=datetime.datetime(2026, 10, 5, 9, 58, 35, 596000, tzinfo=tzlocal()), last_modified_time=datetime.datetime(2026, 10, 5, 9, 58, 35, 670000, tzinfo=tzlocal()), monitoring_schedule_config=MonitoringScheduleConfig(schedule_config=ScheduleConfig(schedule_expression='cron(0 * ? * * *)', data_analysis_start_time=Unassigned(), data_analysis_end_time=Unassigned()), monitoring_job_definition=MonitoringJobDefinition(monitoring_inputs=[MonitoringInput(endpoint_input=EndpointInput(endpoint_name='wine-endpoint', local_path='/opt/ml/processing/input/endpoint', s3_input_mode='File', s3_data_distribution_type='FullyReplicated', features_attribute=Unassigned(), inference_attribute=Unassign

Great job! You can check that your schedule was created by selecting the `SageMaker components and registries` tab on the far left.

In this exercise you configured Model Monitor to watch a simple model. Next, we'll monitor the same deployment for explainability.

__REMINDER:__ Don't leave your model deployed overnight. If you aren't going to follow up with the Clarify exercise within a few hours, use the code below to remove your model:

In [26]:
# monitors = xgb_predictor.list_monitors()
# for monitor in monitors:
#     monitor.delete_monitoring_schedule()

import boto3, time
sm = boto3.client('sagemaker')

for s in sm.list_monitoring_schedules(EndpointName="wine-endpoint")["MonitoringScheduleSummaries"]:
    name = s["MonitoringScheduleName"]
    sm.delete_monitoring_schedule(MonitoringScheduleName=name)
    while True:
        try:
            sm.describe_monitoring_schedule(MonitoringScheduleName=name)
            time.sleep(10)
        except Exception:
            print(name, "gone")
            break

my-wine-endpoint-monitoring-schedule gone


In [27]:
# xgb_predictor.delete_endpoint()

sm.delete_endpoint(EndpointName="wine-endpoint")
while True:
    try:
        sm.describe_endpoint(EndpointName="wine-endpoint")
        time.sleep(10)
    except Exception:
        print("endpoint gone")
        break

sm.delete_endpoint_config(EndpointConfigName="wine-config")
sm.delete_model(ModelName="wine-model")
print("done")

endpoint gone
done


## Clarify

For the last exercise we'll deploy an explainability monitor using Clarify. We're going to use the model that you deployed in the last exercise, but if you cleaned up your deployments from the previous exercise, that's ok! You can rerun the deployment from the previous exercise up to the point where we deployed our model. It'll look like this:

```python
xgb_predictor = model.deploy(
    initial_instance_count=1, instance_type='ml.m4.xlarge',
    data_capture_config=data_capture_config
)
```

Once your model is deployed, you can come back here. _REMINDER_: you need to clean up your deployment, don't leave it running overnight. We'll provide some code at the end to delete your deployment.

## Prep

We'll begin by reloading our data from the previous exercise.

In [22]:
data = datasets.load_wine()
df = pd.DataFrame(data['data'])
df.columns = data['feature_names']
df.rename(columns = {'od280/od315_of_diluted_wines':'od280_od315_of_diluted_wines'}, inplace=True)

We also need to put the target variable in the first column per the docs for our chosen algorithm: https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html

In [23]:
df["TARGET"] = data['target']
df.set_index(df.pop('TARGET'), inplace=True)
df.reset_index(inplace=True)

In [24]:
df.head()

,TARGET,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280_od315_of_diluted_wines,proline
0,0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,0,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,0,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,0,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,0,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


Now we'll upload the data to S3 as train and validation data:

In [25]:
# delimiter = int(len(df)/2)
# train, test = df.iloc[delimiter:], df.iloc[:delimiter]

# train.to_csv("train.csv", header=False, index=False)
# test.to_csv("validation.csv", header=False, index=False)

# val_location = session.upload_data('./validation.csv', key_prefix="data")
# train_location = session.upload_data('./train.csv', key_prefix="data")

# s3_input_train = sagemaker.inputs.TrainingInput(s3_data=train_location, content_type='csv')
# s3_input_validation = sagemaker.inputs.TrainingInput(s3_data=val_location, content_type='csv')

from sagemaker.core.training.configs import InputData  # Added since sagemaker.inputs.TrainingInput() is deprecated

delimiter = int(len(df)/2)
train, test = df.iloc[delimiter:], df.iloc[:delimiter]

train.to_csv("train.csv", header=False, index=False)
test.to_csv("validation.csv", header=False, index=False)

val_location = session.upload_data('./validation.csv', key_prefix="data")
train_location = session.upload_data('./train.csv', key_prefix="data")

# s3_input_train = sagemaker.inputs.TrainingInput(s3_data=train_location, content_type='csv')
# s3_input_validation = sagemaker.inputs.TrainingInput(s3_data=val_location, content_type='csv')

s3_input_train = InputData(channel_name="train", data_source=train_location, content_type="text/csv")
s3_input_validation = InputData(channel_name="validation", data_source=val_location, content_type="text/csv")

Great! Our data is staged and our model is deployed - let's monitor it for explainability. We need to define three config objects, the `SHAPConfig`, the `ModelConfig`, and the `ExplainabilityAnalysisConfig`. Below, we provide the `SHAPConfig`.

In [49]:
# # Debug - used to delete the ModelConfig and ExplainabilityAnalysisConfig to rerun the cells below

# import boto3
# sm = boto3.client('sagemaker')
# try:
#     sm.delete_monitoring_schedule(MonitoringScheduleName="explainability-schedule")
# except Exception as e:
#     print("schedule:", e)

# try:
#     ModelExplainabilityJobDefinition.get(
#         job_definition_name="wine-explainability-def"
#     ).delete()
# except Exception as e:
#     print("delete:", e)

In [50]:
# shap_config = sagemaker.clarify.SHAPConfig(
#     baseline=[train.mean().astype(int).to_list()[1:]],
#     num_samples=int(train.size),
#     agg_method="mean_abs",
#     save_local_shap_values=False,
# )

# 1 - Build and upload the analysis config
import json, boto3

analysis_config = {
    "methods": {
        "shap": {
            "baseline": [train.mean().astype(int).to_list()[1:]],
            "num_samples": int(train.size),
            "agg_method": "mean_abs",
            "save_local_shap_values": False,
        }
    },
    "headers": train.columns.to_list()[1:],
    "predictor": {
        "model_name": "wine-model",
        "instance_type": "ml.m5.xlarge",
        "initial_instance_count": 1,
        "content_type": "text/csv",
        "accept_type": "text/csv",
    },
}

explainability_uri = f"s3://{bucket}/model_explainability"
config_key = "model_explainability/analysis_config.json"
config_uri = f"s3://{bucket}/{config_key}"

boto3.client("s3").put_object(
    Bucket=bucket, Key=config_key, Body=json.dumps(analysis_config).encode()
)


{'ResponseMetadata': {'RequestId': 'XAAJV1T5QN425065',
  'HostId': '5+JzMjFFyokFKSZJb1M/vALAvnBYtrFHfV+SZAClAjbFR3b74WfqmjrRi3H3LD4s17l/hQsGOFw=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '5+JzMjFFyokFKSZJb1M/vALAvnBYtrFHfV+SZAClAjbFR3b74WfqmjrRi3H3LD4s17l/hQsGOFw=',
   'x-amz-request-id': 'XAAJV1T5QN425065',
   'date': 'Tue, 06 Oct 2026 11:37:08 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"78a880222d9e8b3e5e4232f10998c093"',
   'x-amz-checksum-crc32': 'D51S4w==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"78a880222d9e8b3e5e4232f10998c093"',
 'ChecksumCRC32': 'D51S4w==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

Next up, fill in the blanks to define the `ModelConfig` and `ExplainabilityAnalysisConfig`.

In [51]:
# TODO
# model_config = sagemaker.clarify.ModelConfig(
# # ...
# )

# analysis_config = sagemaker.model_monitor.ExplainabilityAnalysisConfig(
# # ...
#     )

# 2 - Create the schedule
# Same MonitoringSchedule shape as the data-quality one, with two differences: the Clarify image instead of the analyzer, and a second monitoring input pointing at the config file.

from sagemaker.core.resources import ModelExplainabilityJobDefinition, MonitoringSchedule
from sagemaker.core.shapes import (
    ModelExplainabilityAppSpecification, ModelExplainabilityJobInput,
    MonitoringOutputConfig, MonitoringOutput, MonitoringS3Output,
    MonitoringResources, MonitoringClusterConfig, MonitoringStoppingCondition,
    EndpointInput, MonitoringScheduleConfig, ScheduleConfig,
)

clarify_image = f"205585389593.dkr.ecr.{region}.amazonaws.com/sagemaker-clarify-processing:1.0"

ModelExplainabilityJobDefinition.create(
    job_definition_name="wine-explainability-def",
    model_explainability_app_specification=ModelExplainabilityAppSpecification(
        image_uri=clarify_image,
        config_uri=config_uri,
    ),
    model_explainability_job_input=ModelExplainabilityJobInput(
        endpoint_input=EndpointInput(
            endpoint_name="wine-endpoint",
            local_path="/opt/ml/processing/input/endpoint",
        )
    ),
    model_explainability_job_output_config=MonitoringOutputConfig(
        monitoring_outputs=[MonitoringOutput(
            s3_output=MonitoringS3Output(
                s3_uri=explainability_uri,
                local_path="/opt/ml/processing/output",
            )
        )]
    ),
    job_resources=MonitoringResources(
        cluster_config=MonitoringClusterConfig(
            instance_count=1,
            instance_type="ml.m5.xlarge",
            volume_size_in_gb=20,
        )
    ),
    stopping_condition=MonitoringStoppingCondition(max_runtime_in_seconds=1800),
    role_arn=role,
)

MonitoringSchedule.create(
    monitoring_schedule_name="explainability-schedule",
    monitoring_schedule_config=MonitoringScheduleConfig(
        schedule_config=ScheduleConfig(schedule_expression="cron(0 * ? * * *)"),
        monitoring_job_definition_name="wine-explainability-def",
        monitoring_type="ModelExplainability",
    ),
)

[10/06/26 11:37:09] INFO     Creating model_explainability_job_definition resource.              ]8;id=14744305;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744306;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#22226\22226]8;;\

                    INFO     Creating monitoring_schedule resource.                              ]8;id=14744311;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=14744312;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#24562\24562]8;;\

MonitoringSchedule(monitoring_schedule_name='explainability-schedule', monitoring_schedule_arn='arn:aws:sagemaker:us-east-1:864107284170:monitoring-schedule/explainability-schedule', monitoring_schedule_status='Pending', monitoring_type='ModelExplainability', failure_reason=Unassigned(), creation_time=datetime.datetime(2026, 10, 6, 11, 37, 9, 822000, tzinfo=tzlocal()), last_modified_time=datetime.datetime(2026, 10, 6, 11, 37, 9, 885000, tzinfo=tzlocal()), monitoring_schedule_config=MonitoringScheduleConfig(schedule_config=ScheduleConfig(schedule_expression='cron(0 * ? * * *)', data_analysis_start_time=Unassigned(), data_analysis_end_time=Unassigned()), monitoring_job_definition=Unassigned(), monitoring_job_definition_name='wine-explainability-def', monitoring_type='ModelExplainability'), endpoint_name='wine-endpoint', last_monitoring_execution_summary=MonitoringExecutionSummary(monitoring_schedule_name='explainability-schedule', scheduled_time=datetime.datetime(2026, 10, 6, 11, 0, tzin

Before we apply our config, we need to create the monitor object. This is what we'll apply all our config to.

In [52]:
# model_explainability_monitor = sagemaker.model_monitor.ModelExplainabilityMonitor(
#     role=role,
#     sagemaker_session=session,
#     max_runtime_in_seconds=1800,
# )

Everything's ready! Below, create a monitoring schedule using the configs we created. Set the schedule to run _daily_.

In [53]:
# # TODO 
# from sagemaker.model_monitor import CronExpressionGenerator


# explainability_uri =
# model_explainability_monitor.create_monitoring_schedule(
# # ...
# )

Way to go! You can check that your schedule was created by selecting the `SageMaker components and registries` tab on the far left.

In this exercise you deployed a monitor for explainability to your SageMaker endpoint. This is the last exercise - you'll apply these learnings again in your Project at the end of the course.



__REMINDER:__ Don't leave your model deployed overnight. Use the code below to remove your model:

In [54]:
# monitors = xgb_predictor.list_monitors()
# for monitor in monitors:
#     monitor.delete_monitoring_schedule()

In [55]:
# List monitoring schedules
import boto3
sm = boto3.client('sagemaker')

for s in sm.list_monitoring_schedules()["MonitoringScheduleSummaries"]:
    print(s["MonitoringScheduleName"], s["MonitoringScheduleStatus"],
          s.get("MonitoringType"), s.get("EndpointName"))

explainability-schedule Scheduled ModelExplainability wine-endpoint


In [56]:
# Same as code block above, but filtering by Endpoint
sm.list_monitoring_schedules(EndpointName="wine-endpoint")

{'MonitoringScheduleSummaries': [{'MonitoringScheduleName': 'explainability-schedule',
   'MonitoringScheduleArn': 'arn:aws:sagemaker:us-east-1:864107284170:monitoring-schedule/explainability-schedule',
   'CreationTime': datetime.datetime(2026, 10, 6, 11, 37, 9, 822000, tzinfo=tzlocal()),
   'LastModifiedTime': datetime.datetime(2026, 10, 6, 11, 37, 14, 265000, tzinfo=tzlocal()),
   'MonitoringScheduleStatus': 'Scheduled',
   'EndpointName': 'wine-endpoint',
   'MonitoringJobDefinitionName': 'wine-explainability-def',
   'MonitoringType': 'ModelExplainability'}],
 'ResponseMetadata': {'RequestId': '1f1adc76-acbc-47e0-a572-245a1d27974c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '1f1adc76-acbc-47e0-a572-245a1d27974c',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosni

In [57]:
# The job definitions are listed separately, since they exist independently of any schedule:
print(sm.list_model_explainability_job_definitions()["JobDefinitionSummaries"])
print(sm.list_data_quality_job_definitions()["JobDefinitionSummaries"])

[{'MonitoringJobDefinitionName': 'wine-explainability-def', 'MonitoringJobDefinitionArn': 'arn:aws:sagemaker:us-east-1:864107284170:model-explainability-job-definition/wine-explainability-def', 'CreationTime': datetime.datetime(2026, 10, 6, 11, 37, 9, 384000, tzinfo=tzlocal()), 'EndpointName': 'wine-endpoint'}]
[]


In [58]:
# xgb_predictor.delete_endpoint()


import boto3, time
sm = boto3.client('sagemaker')


def wait_gone(describe, label, **kw):
    while True:
        try:
            describe(**kw)
            time.sleep(10)
        except Exception:
            print(f"{label} gone")
            return


# 1. All schedules on this endpoint
for s in sm.list_monitoring_schedules(EndpointName="wine-endpoint")["MonitoringScheduleSummaries"]:
    name = s["MonitoringScheduleName"]
    try:
        sm.delete_monitoring_schedule(MonitoringScheduleName=name)
        print(name, "deleting...")
    except Exception as e:
        print(name, e)
    wait_gone(sm.describe_monitoring_schedule, name, MonitoringScheduleName=name)

# 2. Job definitions (only deletable once no schedule references them)
for d in sm.list_model_explainability_job_definitions()["JobDefinitionSummaries"]:
    try:
        sm.delete_model_explainability_job_definition(
            JobDefinitionName=d["MonitoringJobDefinitionName"])
        print("deleted", d["MonitoringJobDefinitionName"])
    except Exception as e:
        print(d["MonitoringJobDefinitionName"], e)

for d in sm.list_data_quality_job_definitions()["JobDefinitionSummaries"]:
    try:
        sm.delete_data_quality_job_definition(
            JobDefinitionName=d["MonitoringJobDefinitionName"])
        print("deleted", d["MonitoringJobDefinitionName"])
    except Exception as e:
        print(d["MonitoringJobDefinitionName"], e)

# 3. Endpoint — the thing that bills continuously
try:
    sm.delete_endpoint(EndpointName="wine-endpoint")
    print("endpoint deleting...")
except Exception as e:
    print("endpoint:", e)

wait_gone(sm.describe_endpoint, "endpoint", EndpointName="wine-endpoint")

# 4. Config and model
for fn, kw in [
    (sm.delete_endpoint_config, {"EndpointConfigName": "wine-config"}),
    (sm.delete_model, {"ModelName": "wine-model"}),
]:
    try:
        fn(**kw)
        print("deleted", kw)
    except Exception as e:
        print(kw, e)

print("done")

explainability-schedule deleting...
explainability-schedule gone
deleted wine-explainability-def
endpoint deleting...
endpoint gone
deleted {'EndpointConfigName': 'wine-config'}
deleted {'ModelName': 'wine-model'}
done
